In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from scipy.stats import spearmanr

matplotlib.rcParams['font.family']      = 'sans-serif'
matplotlib.rcParams['font.sans-serif']  = ['Helvetica Neue', 'Helvetica', 'Arial']
matplotlib.rcParams['hatch.linewidth']  = 2.5

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import SPE1_PICKLE_ROOT, CELL_IDS, DICT_CELL_TYPE

SPE1_PKL   = SPE1_PICKLE_ROOT
PVC6_PKL   = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/pvc6_pickles'

In [ ]:
# ── Run-control ────────────────────────────────────────────────────────────────
FORCE_RERUN = False

## Panel A — Algorithm schematic

In [ ]:
# Load a priority spe-1 cell spike fit (c21: PC, large waveform variability)
_sp_path = os.path.join(SPE1_PKL, 'spike_fit_pickles', 'c21_spike_fit.pkl')
with open(_sp_path, 'rb') as _f:
    sp_sch = pickle.load(_f)

# Generate fit arrays
sp_sch.gen_fit(ramp=True, exp=True)

# Build time axis (in ms, centred at peak = 0)
_n = len(sp_sch.spikes[0])
_t_full = np.arange(_n) / (sp_sch.fs / 1000)   # ms
_peak_idx = int(sp_sch.indices[0, 3])            # peak index of first spike
_t_full   = _t_full - _t_full[_peak_idx]         # centre at peak

# Pick a representative spike: closest-to-median peak_amp with good fits
_df = sp_sch.df_features.copy()
_med = _df['peak_amp'].median()
_i  = (_df['peak_amp'] - _med).abs().idxmin()   # original (filtered) index
_i_arr = np.where(np.arange(len(sp_sch.spikes)) == _i)[0]
if len(_i_arr) == 0:
    _i_arr = [_i]
_si = _i_arr[0]   # array index into sp_sch.spikes

print(f'Selected spike index {_si}  peak_amp={sp_sch.peak_amp[_si]:.1f} mV')
print(f'r2_exp={sp_sch.r_squared_exp[_si]:.3f}  r2_ramp={sp_sch.r_squared_ramp[_si]:.3f}')

In [ ]:
def plot_algorithm_schematic(sp, si, t_full, ax,
                              col_ramp='#4C72B0', col_exp='#C44E52',
                              col_wf='#2d2d2d', lw_wf=2.5, lw_fit=2.2,
                              fs_label=11, fs_annot=9):
    """Draw annotated single-spike schematic on ax."""
    inds  = sp.indices[si]   # [ramp_start, inflection, rise, peak, decay, exp_start, exp_end]
    idx_ramp_start, idx_inflection, idx_rise, idx_peak, idx_decay, idx_exp_start, idx_exp_end = inds

    wf = sp.spikes[si]

    # ── Full waveform ───────────────────────────────────────────────────────
    ax.plot(t_full, wf, color=col_wf, lw=lw_wf, zorder=3)

    # ── Ramp segment highlight + fit ────────────────────────────────────────
    _r_sl = slice(idx_ramp_start, idx_inflection)
    ax.plot(t_full[_r_sl], wf[_r_sl], color=col_ramp, lw=lw_wf + 0.5, zorder=4, alpha=0.85)
    if sp.fit_ramp is not None and not np.any(np.isnan(sp.fit_ramp[si])):
        ax.plot(t_full[_r_sl], sp.fit_ramp[si], color=col_ramp,
                lw=lw_fit, ls='--', zorder=5, label='Ramp fit')

    # ── Exp decay segment highlight + fit ──────────────────────────────────
    _e_sl = slice(idx_exp_start, idx_exp_end)
    ax.plot(t_full[_e_sl], wf[_e_sl], color=col_exp, lw=lw_wf + 0.5, zorder=4, alpha=0.85)
    if sp.fit_exp is not None and not np.any(np.isnan(sp.fit_exp[si])):
        ax.plot(t_full[_e_sl], sp.fit_exp[si], color=col_exp,
                lw=lw_fit, ls='--', zorder=5, label='Exp. decay fit')

    # ── Key control points ──────────────────────────────────────────────────
    _pts = {
        'inflection': (t_full[idx_inflection], wf[idx_inflection]),
        'peak':       (t_full[idx_peak],       wf[idx_peak]),
        'exp_start':  (t_full[idx_exp_start],  wf[idx_exp_start]),
    }
    for name, (tx, ty) in _pts.items():
        ax.scatter([tx], [ty], s=55, color='white', edgecolors='k',
                   linewidths=1.3, zorder=6)

    # ── Annotations ─────────────────────────────────────────────────────────
    _t_infl = t_full[idx_inflection]
    _t_peak = t_full[idx_peak]
    _t_ramp_start = t_full[idx_ramp_start]
    _t_exp_start  = t_full[idx_exp_start]

    _v_infl  = wf[idx_inflection]
    _v_peak  = wf[idx_peak]
    _v_base  = wf[idx_ramp_start]          # baseline near ramp start
    _v_exp_start = wf[idx_exp_start]

    ax_ymin = ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else _v_base

    arrowprops = dict(arrowstyle='->', color='k', lw=1.0)

    # ramp_amp: vertical span from baseline to inflection
    ax.annotate('', xy=(_t_infl, _v_infl), xytext=(_t_infl, _v_base),
                arrowprops=dict(arrowstyle='<->', color=col_ramp, lw=1.5))
    ax.text(_t_infl - 0.05, (_v_infl + _v_base) / 2, 'ramp\namp',
            ha='right', va='center', fontsize=fs_annot, color=col_ramp, fontweight='bold')

    # inflection_time: horizontal span from ramp_start to inflection
    _y_infl_time = _v_base - 5
    ax.annotate('', xy=(_t_infl, _y_infl_time),
                xytext=(_t_ramp_start, _y_infl_time),
                arrowprops=dict(arrowstyle='<->', color=col_ramp, lw=1.5))
    ax.text((_t_infl + _t_ramp_start) / 2, _y_infl_time - 3,
            'infl. time', ha='center', va='top', fontsize=fs_annot, color=col_ramp, fontweight='bold')

    # peak_amp: vertical from baseline to peak
    ax.annotate('', xy=(_t_peak + 0.15, _v_peak), xytext=(_t_peak + 0.15, _v_base),
                arrowprops=dict(arrowstyle='<->', color='k', lw=1.5))
    ax.text(_t_peak + 0.22, (_v_peak + _v_base) / 2, 'peak\namp',
            ha='left', va='center', fontsize=fs_annot, fontweight='bold')

    # exp_lambda label on exp decay segment
    _t_mid_exp = (_t_exp_start + t_full[idx_exp_end - 1]) / 2
    _v_mid_exp = wf[int((idx_exp_start + idx_exp_end) / 2)]
    ax.text(_t_mid_exp + 0.25, _v_mid_exp, 'exp λ',
            ha='left', va='center', fontsize=fs_annot, color=col_exp, fontweight='bold')

    # peak_width: horizontal line at half-max
    _half = (_v_peak + _v_base) / 2
    # find crossing indices for peak_width visualization (approximate)
    _t_rise_pt  = t_full[idx_rise]
    _t_decay_pt = t_full[idx_decay]
    ax.plot([_t_rise_pt, _t_decay_pt], [_half, _half], color='k', lw=1.2, ls=':')
    ax.text((_t_rise_pt + _t_decay_pt) / 2, _half + 3,
            'peak width', ha='center', va='bottom', fontsize=fs_annot, fontweight='bold')

    # Styling
    ax.set_xlabel('Time from peak (ms)', fontsize=fs_label, fontweight='bold')
    ax.set_ylabel('Voltage (mV)', fontsize=fs_label, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=fs_annot + 1)
    for sp_ in ['bottom', 'left']:
        ax.spines[sp_].set_linewidth(1.5)

    # Clip x to a tight window around the spike
    _t_lo = max(t_full[idx_ramp_start] - 0.5, t_full[0])
    _t_hi = min(t_full[idx_exp_end - 1] + 0.5, t_full[-1])
    ax.set_xlim(_t_lo, _t_hi)

    # Legend (fits)
    ax.legend(frameon=False, fontsize=fs_annot, loc='upper right')


fig_sch, ax_sch = plt.subplots(figsize=(7, 4))
plot_algorithm_schematic(sp_sch, _si, _t_full, ax_sch)
plt.tight_layout()
plt.show()

## Panel B — Intra-spike feature correlations

In [ ]:
FEAT_COLS = ['peak_amp', 'peak_width', 'peak_sharpness',
             'inflection_amp', 'inflection_time', 'exp_lambda', 'ramp_amp']
FEAT_LABS = ['Peak\nAmp', 'Peak\nWidth', 'Peak\nSharp.',
             'Infl.\nAmp', 'Infl.\nTime', 'Exp\nλ', 'Ramp\nAmp']

def cell_spearman_matrix(df, feat_cols):
    """Compute pairwise Spearman correlation matrix for one cell's df."""
    n = len(feat_cols)
    mat = np.full((n, n), np.nan)
    sub = df[feat_cols].dropna()
    if len(sub) < 5:
        return mat
    for i, c1 in enumerate(feat_cols):
        for j, c2 in enumerate(feat_cols):
            if i == j:
                mat[i, j] = 1.0
            else:
                r, _ = spearmanr(sub[c1], sub[c2])
                mat[i, j] = r
    return mat


# ── spe-1: average correlation matrix across all cells ────────────────────────
_cluster_pkl_dir = os.path.join(SPE1_PKL, 'cluster_pickles')
_spe1_mats = []
_spe1_cell_ids = []
for _cid in CELL_IDS:
    _p = os.path.join(_cluster_pkl_dir, f'{_cid}_cluster_df.pkl')
    if not os.path.exists(_p):
        continue
    with open(_p, 'rb') as _f:
        _df = pickle.load(_f)
    _available = [c for c in FEAT_COLS if c in _df.columns]
    if len(_available) < len(FEAT_COLS):
        print(f'{_cid}: missing columns {set(FEAT_COLS) - set(_available)}')
        continue
    _mat = cell_spearman_matrix(_df, FEAT_COLS)
    _spe1_mats.append(_mat)
    _spe1_cell_ids.append(_cid)

spe1_mean_corr = np.nanmean(np.stack(_spe1_mats), axis=0)
print(f'spe-1: {len(_spe1_mats)} cells used')

# ── pvc-6: average correlation matrix across cells 1 and 2 ───────────────────
_pvc6_mats = []
for _suffix in ['_c1', '_c2']:
    _pkl_name = 'df_all_c1.pkl' if _suffix == '_c1' else 'df_all_c2.pkl'
    _p = os.path.join(PVC6_PKL, _pkl_name)
    if not os.path.exists(_p):
        continue
    with open(_p, 'rb') as _f:
        _df = pickle.load(_f)
    _mat = cell_spearman_matrix(_df, FEAT_COLS)
    _pvc6_mats.append(_mat)

pvc6_mean_corr = np.nanmean(np.stack(_pvc6_mats), axis=0)
print(f'pvc-6: {len(_pvc6_mats)} cells used')

In [ ]:
def plot_corr_heatmap(mat, feat_labs, ax, title='', vmin=-1, vmax=1,
                      cmap='RdBu_r', fs_label=10, fs_tick=8, fs_title=11,
                      lw_sp=1.5):
    import matplotlib.colors as mcolors
    n = len(feat_labs)
    # mask diagonal for cleaner look
    _mat = mat.copy()
    np.fill_diagonal(_mat, np.nan)

    im = ax.imshow(_mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

    # Annotate cells
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            _v = mat[i, j]
            _fc = 'white' if abs(_v) > 0.55 else 'black'
            ax.text(j, i, f'{_v:.2f}', ha='center', va='center',
                    fontsize=fs_tick - 1, color=_fc, fontweight='bold')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(feat_labs, fontsize=fs_tick, fontweight='bold', rotation=0)
    ax.set_yticklabels(feat_labs, fontsize=fs_tick, fontweight='bold')
    ax.set_title(title, fontsize=fs_title, fontweight='bold', pad=10)

    for sp_ in ax.spines.values():
        sp_.set_linewidth(lw_sp)

    return im


fig_corr, axes_corr = plt.subplots(1, 2, figsize=(12, 5))
im1 = plot_corr_heatmap(spe1_mean_corr, FEAT_LABS, axes_corr[0],
                         title=f'spe-1 (n={len(_spe1_mats)} cells)')
im2 = plot_corr_heatmap(pvc6_mean_corr, FEAT_LABS, axes_corr[1],
                         title='pvc-6 (n=2 cells)')

# shared colorbar
from mpl_toolkits.axes_grid1 import make_axes_locatable
_divider = make_axes_locatable(axes_corr[1])
_cax = _divider.append_axes('right', size='5%', pad=0.1)
cbar = plt.colorbar(im2, cax=_cax)
cbar.set_label('Spearman r', fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

## Panel C — R² fit distributions (spe-1)

In [ ]:
_fit_pkl_dir = os.path.join(SPE1_PKL, 'spike_fit_pickles')

_r2_records = []
for _cid in CELL_IDS:
    _p = os.path.join(_fit_pkl_dir, f'{_cid}_spike_fit.pkl')
    if not os.path.exists(_p):
        continue
    with open(_p, 'rb') as _f:
        _sp = pickle.load(_f)
    if _sp.r_squared_exp is None or _sp.r_squared_ramp is None:
        # Generate fits to populate R²
        _sp.gen_fit(ramp=True, exp=True)

    _cnum = int(_cid.replace('c', ''))
    _ctype = DICT_CELL_TYPE.get(_cnum, 'PC')

    _r2_exp  = np.asarray(_sp.r_squared_exp,  dtype=float)
    _r2_ramp = np.asarray(_sp.r_squared_ramp, dtype=float)

    # Per-cell median (one value per cell)
    _r2_records.append({
        'cell_id':     _cid,
        'cell_type':   _ctype,
        'median_r2_exp':  float(np.nanmedian(_r2_exp)),
        'median_r2_ramp': float(np.nanmedian(_r2_ramp)),
        'r2_exp_all':  _r2_exp[~np.isnan(_r2_exp)],
        'r2_ramp_all': _r2_ramp[~np.isnan(_r2_ramp)],
    })

df_r2 = pd.DataFrame(_r2_records)
print(f'Loaded {len(df_r2)} cells')
print(df_r2[['cell_id', 'cell_type', 'median_r2_exp', 'median_r2_ramp']].to_string())

In [ ]:
from matplotlib.patches import Patch

_PC_COL = '#4878D0'
_IN_COL = '#EE854A'
_TYPE_COLS = {'PC': _PC_COL, 'IN': _IN_COL}
_FS_AX  = 11
_FS_TK  = 9
_LW_SP  = 1.5

def plot_r2_violins(df_r2, r2_key, title, ax, bins=20):
    """Per-cell violin plot of R² distribution, coloured by cell type."""
    _records = df_r2.sort_values('cell_type').reset_index(drop=True)
    _x_pos = 0
    _xtick_pos  = []
    _xtick_labs = []

    for _, row in _records.iterrows():
        _vals = row[f'{r2_key}_all']
        _col  = _TYPE_COLS.get(row['cell_type'], _PC_COL)
        if len(_vals) > 3:
            _vp = ax.violinplot(_vals, positions=[_x_pos], widths=0.7,
                                showmedians=True, showextrema=False)
            for _part in _vp['bodies']:
                _part.set_facecolor(_col)
                _part.set_alpha(0.65)
                _part.set_edgecolor('none')
            _vp['cmedians'].set_color('white')
            _vp['cmedians'].set_linewidth(1.5)
        _xtick_pos.append(_x_pos)
        _xtick_labs.append(row['cell_id'])
        _x_pos += 1

    ax.set_xticks(_xtick_pos)
    ax.set_xticklabels(_xtick_labs, rotation=90, fontsize=_FS_TK - 1, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('R²', fontsize=_FS_AX, fontweight='bold')
    ax.set_title(title, fontsize=_FS_AX, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for sp_ in ['bottom', 'left']:
        ax.spines[sp_].set_linewidth(_LW_SP)
    ax.tick_params(labelsize=_FS_TK)

    # Add a dashed horizontal reference at 0.5
    ax.axhline(0.5, color='k', ls='--', lw=1.0, alpha=0.5)


fig_r2, (ax_r2_exp, ax_r2_ramp) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
plot_r2_violins(df_r2, 'r2_exp',  'Exponential decay fit R²', ax_r2_exp)
plot_r2_violins(df_r2, 'r2_ramp', 'Ramp fit R²',              ax_r2_ramp)

ax_r2_ramp.set_ylabel('')

# Legend: PC vs IN
_leg_els = [
    Patch(facecolor=_PC_COL, alpha=0.65, label='Pyramidal (PC)'),
    Patch(facecolor=_IN_COL, alpha=0.65, label='Interneuron (IN)'),
]
fig_r2.legend(handles=_leg_els, loc='upper right', frameon=False,
              fontsize=_FS_AX, bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()
plt.show()

## Assemble final figure

In [ ]:
matplotlib.rcParams['font.family']      = 'sans-serif'
matplotlib.rcParams['font.sans-serif']  = ['Helvetica Neue', 'Helvetica', 'Arial']

_FS_PANEL = 16   # panel label
_FS_TITLE = 13
_FS_LABEL = 11
_FS_TICK  = 9
_LW_SP    = 1.8

fig = plt.figure(figsize=(18, 14))
gs_top = gridspec.GridSpec(2, 1, figure=fig, hspace=0.45,
                            top=0.97, bottom=0.05, left=0.07, right=0.97)

# Top row: Panel A (schematic, left) + Panel B (correlations, right)
gs_top_row = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_top[0], wspace=0.45,
    width_ratios=[1, 2]
)

ax_a   = fig.add_subplot(gs_top_row[0])   # Algorithm schematic
gs_b   = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_top_row[1], wspace=0.35
)
ax_b1  = fig.add_subplot(gs_b[0])         # spe-1 correlations
ax_b2  = fig.add_subplot(gs_b[1])         # pvc-6 correlations

# Bottom row: Panel C (R² — exp and ramp side by side)
gs_bot = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs_top[1], wspace=0.25
)
ax_c1  = fig.add_subplot(gs_bot[0])       # R² exp decay
ax_c2  = fig.add_subplot(gs_bot[1])       # R² ramp

# ── A: Algorithm schematic ────────────────────────────────────────────────────
plot_algorithm_schematic(sp_sch, _si, _t_full, ax_a,
                          fs_label=_FS_LABEL, fs_annot=_FS_TICK)
ax_a.set_title('Spike waveform\nparameterization', fontsize=_FS_TITLE,
                fontweight='bold', pad=10)

# ── B: Intra-spike correlations ───────────────────────────────────────────────
im1 = plot_corr_heatmap(spe1_mean_corr, FEAT_LABS, ax_b1,
                         title=f'spe-1 (n={len(_spe1_mats)} cells)',
                         fs_label=_FS_LABEL, fs_tick=_FS_TICK, fs_title=_FS_TITLE,
                         lw_sp=_LW_SP)
im2 = plot_corr_heatmap(pvc6_mean_corr, FEAT_LABS, ax_b2,
                         title='pvc-6 (n=2 cells)',
                         fs_label=_FS_LABEL, fs_tick=_FS_TICK, fs_title=_FS_TITLE,
                         lw_sp=_LW_SP)

# Shared colorbar for B
_divider = make_axes_locatable(ax_b2)
_cax2 = _divider.append_axes('right', size='5%', pad=0.08)
_cbar = plt.colorbar(im2, cax=_cax2)
_cbar.set_label('Spearman r', fontsize=_FS_TICK + 1, fontweight='bold')
_cbar.ax.tick_params(labelsize=_FS_TICK)

# ── C: R² distributions ───────────────────────────────────────────────────────
plot_r2_violins(df_r2, 'r2_exp',  'Exponential decay fit R²  (spe-1)', ax_c1)
plot_r2_violins(df_r2, 'r2_ramp', 'Ramp fit R²  (spe-1)',              ax_c2)
ax_c2.set_ylabel('')

_leg_els = [
    Patch(facecolor=_PC_COL, alpha=0.65, label='Pyramidal (PC)'),
    Patch(facecolor=_IN_COL, alpha=0.65, label='Interneuron (IN)'),
]
ax_c2.legend(handles=_leg_els, frameon=False, fontsize=_FS_TICK + 1,
             loc='lower right')

# ── Panel labels ──────────────────────────────────────────────────────────────
for _ax, _lbl in [(ax_a, 'A'), (ax_b1, 'B'), (ax_c1, 'C')]:
    _ax.text(-0.15, 1.07, _lbl, transform=_ax.transAxes,
             fontsize=_FS_PANEL, fontweight='bold', va='top', ha='left')

# Save
plt.savefig('supp_algorithm_correlations_r2.pdf', bbox_inches='tight', dpi=300)
plt.savefig('supp_algorithm_correlations_r2.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')